In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

In [0]:
CATALOG = 'retail_data'
GOLD_SCHEMA = 'gold'
SILVER_SCHEMA = 'silver'
yest_date = datetime.now() - timedelta(days=1)
yest_date = yest_date.strftime("%Y%m%d")
dbutils.widgets.text("file_run", yest_date, "File Run Parameter")
file_run = dbutils.widgets.get("file_run")

In [0]:
spark.sql(
f'''
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.customers
USING DELTA
AS
SELECT
    sha2(mobile_number, 256) AS customer_sk,
    full_name,
    email,
    mobile_number,
    city,
    state,
    zipcode,
    signup_date,
    offline_customer_id,
    online_customer_id,
    file_date,
    source,
    ingestion_ts,
    current_timestamp() AS last_updated_ts
FROM {CATALOG}.{SILVER_SCHEMA}.customers_unified
WHERE 1 = 2;
'''
)

In [0]:
customers_gold = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customers")
customers_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_unified")

In [0]:
spark.sql(
f'''
MERGE INTO {CATALOG}.{GOLD_SCHEMA}.customers t
USING (
    SELECT
        sha2(mobile_number, 256) AS customer_sk,
        full_name,
        email,
        mobile_number,
        city,
        state,
        zipcode,
        signup_date,
        offline_customer_id,
        online_customer_id,
        file_date,
        source,
        ingestion_ts
    FROM {CATALOG}.{SILVER_SCHEMA}.customers_unified
    WHERE file_date = {file_run}
) s
ON t.mobile_number = s.mobile_number

WHEN MATCHED THEN UPDATE SET
    t.customer_sk = s.customer_sk,
    t.full_name = s.full_name,
    t.email = s.email,
    t.city = s.city,
    t.state = s.state,
    t.zipcode = s.zipcode,
    t.signup_date = s.signup_date,
    t.offline_customer_id = s.offline_customer_id,
    t.online_customer_id = s.online_customer_id,
    t.file_date = s.file_date,
    t.source = s.source,
    t.ingestion_ts = s.ingestion_ts,
    t.last_updated_ts = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    customer_sk,
    full_name,
    email,
    mobile_number,
    city,
    state,
    zipcode,
    signup_date,
    offline_customer_id,
    online_customer_id,
    file_date,
    source,
    ingestion_ts,
    last_updated_ts
)
VALUES (
    s.customer_sk,
    s.full_name,
    s.email,
    s.mobile_number,
    s.city,
    s.state,
    s.zipcode,
    s.signup_date,
    s.offline_customer_id,
    s.online_customer_id,
    s.file_date,
    s.source,
    s.ingestion_ts,
    current_timestamp()
);
'''
)